# Asistente de reglas del SRD 5.2.1

Ejecuta el código del repositorio. Requiere el secreto `ANTHROPIC_API_KEY`.

## 1. Preparación

In [ ]:
import os
if os.path.exists('/content/dnd_assistant'):
    !cd /content/dnd_assistant && git pull -q
else:
    !git clone -q https://github.com/MarioGavin/dnd_assistant.git /content/dnd_assistant
%cd /content/dnd_assistant
!pip -q install pymupdf rank-bm25 snowballstemmer anthropic pytest

# La clave de los Secretos pasa al entorno para que la lean los scripts
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

## 2. Tests

In [ ]:
!python -m pytest -q

## 3. Consultas

In [ ]:
!python -m asistente.responder "Tengo 2 niveles de cansancio, ¿qué me pasa al atacar?"

In [ ]:
!python -m asistente.responder "Dame la CA y los PG del contemplador"

In [ ]:
!python -m asistente.responder "Hemos pillado por sorpresa a unos bandidos, ¿pierden su primer turno?"

In [ ]:
# Validador de fichas: sin modelo, sin coste
!python -m asistente.fichas

## 4. Evaluación de la recuperación: BM25

In [ ]:
!python -m asistente.evaluar --solo-recuperacion --sin-vectores
!python -m asistente.evaluar --solo-recuperacion --sin-vectores --casos casos/consultas_contraste.json

## 5. Evaluación completa (consultas y fichas)

Tres ejecuciones por modelo: la API no admite `temperature`, así que la variabilidad se mide repitiendo.

In [ ]:
!python -m asistente.evaluar --modelo claude-haiku-5-5 --repeticiones 3

In [ ]:
!python -m asistente.evaluar --modelo claude-sonnet-5-5 --repeticiones 3

In [ ]:
# Haiku con la pregunta reformulada antes de buscar (DECISIONES.md, fila 14)
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --repeticiones 3

In [ ]:
# La primera es la configuración entregada (reformular + contexto ampliado, filas 14 y 19); las otras, opciones medidas
# contexto ampliado (60 candidatos, 20 fragmentos, 2 de 2014 y tablas del padre) y tablas del apartado padre (fallo 24)
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --contexto-amplio --repeticiones 3
!python -m asistente.evaluar --modelo claude-haiku-5-5 --tablas-padre --repeticiones 3
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --tablas-padre --repeticiones 3

In [ ]:
!python -m asistente.evaluar --tabla

## 6. Evaluación de la recuperación: combinada (opcional, GPU)

In [ ]:
!pip -q install sentence-transformers
!python -m asistente.evaluar --solo-recuperacion

In [ ]:
!python -m asistente.evaluar --solo-recuperacion --modelo-emb BAAI/bge-m3

## 6b. Modelo local: Qwen3 (opcional, GPU)

Antes: Entorno de ejecución → Cambiar tipo de entorno → una GPU (A100 o L4: 16 bits sin cuantizar; T4: el 8B y el 14B en 4 bits), y volver a ejecutar la sección 1. Mismo prompt y mismo contexto que `haiku-5.5` (sin reformular, contexto normal), sin razonamiento. La GPU, el tipo de datos y si se cuantizó quedan en `resumen.carga_local`.

In [ ]:
!pip -q install -U "transformers>=4.51" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Prueba de tiempo con 2 consultas. Si cada una pasa de ~60 s, no lances la evaluación completa
import gc, torch
from asistente import responder as R
for p in ["Mi pícaro se cae desde 12 metros. ¿Cuánto daño recibe?",
          "Dame la CA, los PG y los ataques del contemplador para un encuentro de nivel 5."]:
    r = R.responder(p, modelo=R.MODELO_LOCAL)
    print(r["segundos"], "s ·", r["tokens_entrada"], "+", r["tokens_salida"], "tokens ·", r["stop_reason"])
    print(r["respuesta"][:400], "\n")
R._locales.clear(); gc.collect(); torch.cuda.empty_cache()   # libera la GPU para la celda siguiente

In [ ]:
# Evaluación completa: 1 ejecución de los 20 casos
!python -m asistente.evaluar --modelo Qwen/Qwen3-4B --repeticiones 1

In [ ]:
!python -m asistente.evaluar --tabla

### Qwen3-8B (el modelo del plan)

En una T4 se carga en 4 bits (NF4); en L4 o A100, en 16 bits. La primera vez descarga unos 16 GB.

In [ ]:
!pip -q install -U bitsandbytes
import gc, torch
from asistente import responder as R
for p in ["Mi pícaro se cae desde 12 metros. ¿Cuánto daño recibe?",
          "Dame la CA, los PG y los ataques del contemplador para un encuentro de nivel 5."]:
    r = R.responder(p, modelo="Qwen/Qwen3-8B")
    print(r["segundos"], "s ·", r["tokens_entrada"], "+", r["tokens_salida"], "tokens ·", r["stop_reason"])
    print(r["respuesta"][:400], "\n")
R._locales.clear(); gc.collect(); torch.cuda.empty_cache()   # libera la GPU para la celda siguiente

In [ ]:
!python -m asistente.evaluar --modelo Qwen/Qwen3-8B --repeticiones 1
!python -m asistente.evaluar --tabla

### Qwen3-14B (opcional, solo con A100: ~30 GB en 16 bits)

In [ ]:
!python -m asistente.evaluar --modelo Qwen/Qwen3-14B --repeticiones 1
!python -m asistente.evaluar --tabla

## 7. Resultados

In [ ]:
!zip -qr resultados.zip resultados
from google.colab import files
files.download('resultados.zip')